# Adaptive Fusion TS2Img — Kaggle Stage 2 runner

Notebook command-only dùng để chạy **Stage 2 — Pilot experiment** của repository:

`https://github.com/hoangtnedu/adaptive-fusion-ts2img`

Quy mô Stage 2:

- 12 datasets;
- 3 seeds: 42, 43, 44;
- 7 methods;
- tổng cộng **252 runs**.

## Thiết lập Kaggle trước khi chạy

1. Chọn **Accelerator: GPU**.
2. Bật **Internet** để clone GitHub và tải dữ liệu UCR.
3. Nên chạy theo 3 chunk, mỗi chunk 84 runs.
4. Source code được clone vào `/kaggle/working`; outputs, cache và checkpoints cũng được lưu trong `/kaggle/working`.

Notebook chỉ điều phối lệnh. Logic mô hình, xử lý dữ liệu và đánh giá nằm trong `src/`; tham số thí nghiệm nằm trong `config/`.

## 1. Cấu hình phiên chạy

In [1]:
from pathlib import Path

REPO_URL = "https://github.com/hoangtnedu/adaptive-fusion-ts2img.git"

PROJECT_DIR = Path("/kaggle/working/adaptive-fusion-ts2img")
RUN_ROOT = Path("/kaggle/working/adaptive_fusion_stage2")

DATA_ROOT = PROJECT_DIR / "data" / "UCR"
OUTPUT_ROOT = RUN_ROOT / "outputs"
CACHE_ROOT = RUN_ROOT / "cache"

# "chunk": chạy một phần gồm 4 datasets = 84 runs.
# "full" : chạy toàn bộ Stage 2 = 252 runs.
RUN_MODE = "chunk"

# Chỉ dùng khi RUN_MODE = "chunk". Giá trị hợp lệ: 1, 2 hoặc 3.
CHUNK_ID = 3

# True: xóa source cũ và clone lại commit mới nhất từ GitHub.
# Outputs/cache không bị xóa vì nằm ngoài PROJECT_DIR.
REFRESH_REPO = True

# Nếu đã lưu stage2_resume_bundle.tar.gz từ phiên trước và Add Data vào
# notebook hiện tại, đặt True để tự động khôi phục outputs/cache.
AUTO_RESTORE = True

RUN_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
CACHE_ROOT.mkdir(parents=True, exist_ok=True)

print("Project :", PROJECT_DIR)
print("Data    :", DATA_ROOT)
print("Outputs :", OUTPUT_ROOT)
print("Cache   :", CACHE_ROOT)
print("Mode    :", RUN_MODE)
print("Chunk   :", CHUNK_ID if RUN_MODE == "chunk" else "all")

Project : /kaggle/working/adaptive-fusion-ts2img
Data    : /kaggle/working/adaptive-fusion-ts2img/data/UCR
Outputs : /kaggle/working/adaptive_fusion_stage2/outputs
Cache   : /kaggle/working/adaptive_fusion_stage2/cache
Mode    : chunk
Chunk   : 3


In [2]:
from pathlib import Path
import shutil
import tarfile

INPUT_ROOT = Path("/kaggle/input")

if not AUTO_RESTORE:
    print("AUTO_RESTORE=False, bỏ qua bước khôi phục.")

else:
    print("Các Input hiện có:")
    for item in INPUT_ROOT.iterdir():
        print(" -", item)

    # Trường hợp Kaggle còn giữ nguyên file tar.gz
    bundles = sorted(
        INPUT_ROOT.rglob("stage2_resume_bundle.tar.gz")
    )

    if bundles:
        bundle = bundles[0]
        print("\nTìm thấy bundle:", bundle)

        with tarfile.open(bundle, "r:gz") as tar:
            tar.extractall("/kaggle/working")

        print("Đã giải nén vào /kaggle/working.")

    else:
        print(
            "\nKhông còn file tar.gz. "
            "Kaggle đã tự giải nén thành các file riêng."
        )

        # Tìm thư mục outputs và cache trong mọi Input
        output_candidates = [
            p for p in INPUT_ROOT.rglob("outputs")
            if p.is_dir()
        ]

        cache_candidates = [
            p for p in INPUT_ROOT.rglob("cache")
            if p.is_dir()
        ]

        print("\nCác thư mục outputs tìm thấy:")
        for p in output_candidates:
            print(" -", p)

        print("\nCác thư mục cache tìm thấy:")
        for p in cache_candidates:
            print(" -", p)

        # Ưu tiên outputs có summary hoặc checkpoint
        def output_score(path: Path) -> int:
            return (
                len(list(path.rglob("summary.json"))) * 100
                + len(list(path.rglob("last_checkpoint.pt"))) * 10
                + len(list(path.rglob("best_model.pt"))) * 5
                + len([f for f in path.rglob("*") if f.is_file()])
            )

        if output_candidates:
            source_outputs = max(
                output_candidates,
                key=output_score
            )

            print("\nĐang sao chép outputs từ:")
            print(source_outputs)

            shutil.copytree(
                source_outputs,
                OUTPUT_ROOT,
                dirs_exist_ok=True
            )

            print("Đã khôi phục outputs vào:")
            print(OUTPUT_ROOT)

        else:
            print("\nKhông tìm thấy thư mục outputs.")

        if cache_candidates:
            source_cache = max(
                cache_candidates,
                key=lambda p: len([
                    f for f in p.rglob("*")
                    if f.is_file()
                ])
            )

            print("\nĐang sao chép cache từ:")
            print(source_cache)

            shutil.copytree(
                source_cache,
                CACHE_ROOT,
                dirs_exist_ok=True
            )

            print("Đã khôi phục cache vào:")
            print(CACHE_ROOT)

        else:
            print("\nKhông tìm thấy thư mục cache.")

    print("\nKhôi phục hoàn tất.")

Các Input hiện có:
 - /kaggle/input/datasets

Không còn file tar.gz. Kaggle đã tự giải nén thành các file riêng.

Các thư mục outputs tìm thấy:
 - /kaggle/input/datasets/hoangtn/adaptive-fusion-stage2-resume-v11/adaptive_fusion_stage2/outputs
 - /kaggle/input/datasets/hoangtn/adaptive-fusion-stage2-resume-v11/adaptive-fusion-ts2img/outputs
 - /kaggle/input/datasets/hoangtn/adaptive-fusion-stage2-resume-v11/stage2_resume_bundle/adaptive_fusion_stage2/outputs
 - /kaggle/input/datasets/hoangtn/adaptive-fusion-stage2-resume/adaptive_fusion_stage2/outputs

Các thư mục cache tìm thấy:
 - /kaggle/input/datasets/hoangtn/adaptive-fusion-stage2-resume-v11/adaptive_fusion_stage2/cache
 - /kaggle/input/datasets/hoangtn/adaptive-fusion-stage2-resume-v11/adaptive-fusion-ts2img/cache
 - /kaggle/input/datasets/hoangtn/adaptive-fusion-stage2-resume-v11/stage2_resume_bundle/adaptive_fusion_stage2/cache
 - /kaggle/input/datasets/hoangtn/adaptive-fusion-stage2-resume/adaptive_fusion_stage2/cache

Đang sao

## 2. Khôi phục kết quả từ phiên Kaggle trước — tùy chọn

In [3]:
import shutil
import tarfile

if AUTO_RESTORE:
    bundles = sorted(Path("/kaggle/input").rglob("stage2_resume_bundle.tar.gz"))
    if bundles:
        bundle = bundles[0]
        print("Restoring:", bundle)
        with tarfile.open(bundle, "r:gz") as tar:
            tar.extractall("/kaggle/working")
        print("Restore completed.")
    else:
        print("Không tìm thấy stage2_resume_bundle.tar.gz trong /kaggle/input.")
else:
    print("AUTO_RESTORE=False, bỏ qua bước khôi phục.")

Không tìm thấy stage2_resume_bundle.tar.gz trong /kaggle/input.


## 3. Clone source code từ GitHub

In [4]:
import os
import shutil

os.chdir("/kaggle/working")

if REFRESH_REPO and PROJECT_DIR.exists():
    print("Removing old source folder...")
    shutil.rmtree(PROJECT_DIR)

if not PROJECT_DIR.exists():
    print("Cloning source code from GitHub...")
    !git clone --depth 1 {REPO_URL} {PROJECT_DIR}
else:
    print("Project exists. Pulling latest changes...")
    %cd {PROJECT_DIR}
    !git pull

%cd {PROJECT_DIR}

print("Current directory:")
!pwd

print("Current Git commit:")
!git rev-parse HEAD

Cloning source code from GitHub...
Cloning into '/kaggle/working/adaptive-fusion-ts2img'...
remote: Enumerating objects: 112, done.
remote: Counting objects: 100% (112/112), done.
remote: Compressing objects: 100% (80/80), done.
remote: Total 112 (delta 29), reused 84 (delta 26), pack-reused 0 (from 0)
Receiving objects: 100% (112/112), 140.70 KiB | 5.86 MiB/s, done.
Resolving deltas: 100% (29/29), done.
/kaggle/working/adaptive-fusion-ts2img
Current directory:
/kaggle/working/adaptive-fusion-ts2img
Current Git commit:
cd8b4bc7ad8dda67dc754e835d7956f09d76b0fa


In [5]:
from pathlib import Path
import subprocess
import sys

path = Path(
    "/kaggle/working/adaptive-fusion-ts2img/src/transforms/ts2image.py"
)

if not path.exists():
    raise FileNotFoundError(f"Không tìm thấy file: {path}")

text = path.read_text(encoding="utf-8")


def replace_function(
    source: str,
    function_name: str,
    next_function_name: str,
    new_code: str,
) -> str:
    start = source.index(f"def {function_name}(")
    end = source.index(f"\ndef {next_function_name}(", start)

    return (
        source[:start]
        + new_code.rstrip()
        + "\n\n"
        + source[end + 1:]
    )


new_make_gaf = '''
def make_gaf(
    X: np.ndarray,
    image_size: int = 64,
    method: str = "summation",
    sample_range=(-1, 1),
) -> np.ndarray:
    """Create GAF images safely for both short and long time series.

    For a series shorter than image_size, GAF is first generated at its
    native temporal resolution and then resized to the configured output size.
    """
    from pyts.image import GramianAngularField
    from skimage.transform import resize

    target_size = int(image_size)
    native_size = min(target_size, int(X.shape[1]))

    transformer = GramianAngularField(
        image_size=native_size,
        method=method,
        sample_range=tuple(sample_range) if sample_range is not None else None,
    )

    images = transformer.fit_transform(X)

    if native_size != target_size:
        images = np.stack(
            [
                resize(
                    img,
                    (target_size, target_size),
                    anti_aliasing=True,
                    preserve_range=True,
                ).astype(np.float32)
                for img in images
            ],
            axis=0,
        )

    return minmax_per_image(images)
'''


new_make_mtf = '''
def make_mtf(
    X: np.ndarray,
    image_size: int = 64,
    n_bins: int = 8,
    strategy: str = "quantile",
) -> np.ndarray:
    """Create MTF images safely for both short and long time series."""
    from pyts.image import MarkovTransitionField
    from skimage.transform import resize

    target_size = int(image_size)
    native_size = min(target_size, int(X.shape[1]))

    transformer = MarkovTransitionField(
        image_size=native_size,
        n_bins=n_bins,
        strategy=strategy,
    )

    images = transformer.fit_transform(X)

    if native_size != target_size:
        images = np.stack(
            [
                resize(
                    img,
                    (target_size, target_size),
                    anti_aliasing=True,
                    preserve_range=True,
                ).astype(np.float32)
                for img in images
            ],
            axis=0,
        )

    return minmax_per_image(images)
'''


text = replace_function(
    text,
    "make_gaf",
    "make_mtf",
    new_make_gaf,
)

text = replace_function(
    text,
    "make_mtf",
    "make_rp",
    new_make_mtf,
)

path.write_text(text, encoding="utf-8")

# Kiểm tra cú pháp
subprocess.run(
    [sys.executable, "-m", "py_compile", str(path)],
    check=True,
)

print("Đã sửa GAF và MTF thành công.")
print("File:", path)
print("Kiểm tra cú pháp: OK")

Đã sửa GAF và MTF thành công.
File: /kaggle/working/adaptive-fusion-ts2img/src/transforms/ts2image.py
Kiểm tra cú pháp: OK


## 4. Cài thư viện và kiểm tra source

In [6]:
!python -m pip install -q -r requirements.txt
!python -m compileall src

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 40.5 MB/s eta 0:00:00
Listing 'src'...
Compiling 'src/__init__.py'...
Listing 'src/cli'...
Compiling 'src/cli/__init__.py'...
Compiling 'src/cli/batch_run.py'...
Compiling 'src/cli/benchmark_inference.py'...
Compiling 'src/cli/check_data.py'...
Compiling 'src/cli/collect_results.py'...
Compiling 'src/cli/configure_run.py'...
Compiling 'src/cli/copy_ucr_dataset.py'...
Compiling 'src/cli/download_data.py'...
Compiling 'src/cli/export_paper_tables.py'...
Compiling 'src/cli/make_paper_package.py'...
Compiling 'src/cli/preview_representations.py'...
Compiling 'src/cli/rank_results.py'...
Compiling 'src/cli/show_results.py'...
Compiling 'src/cli/statistical_tests.py'...
Listing 'src/data'...
Compiling 'src/data/__init__.py'...
Compiling 'src/data/ucr_downloader.py'...
Compiling 'src/data/ucr_loader.py'...
Listing 'src/datasets'...
Compiling 'src/datasets/__init__.py'...
Compiling 'src/datasets/multi_rep_dataset.py'...
Compiling 'src/data

In [7]:
import platform
import torch

print("Python :", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA   :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU    :", torch.cuda.get_device_name(0))
    !nvidia-smi
else:
    raise RuntimeError(
        "CUDA không khả dụng. Hãy vào Kaggle Notebook Settings và chọn GPU accelerator."
    )

Python : 3.12.13
PyTorch: 2.10.0+cu128
CUDA   : True
GPU    : Tesla T4
Sat Jul 11 10:59:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8             11W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        | 

## 5. Tải 12 datasets của Stage 2

In [8]:
!python -m src.cli.download_data \
  --config config/suites/stage2_pilot_12datasets_3seeds_all_methods.yaml \
  --data-root {DATA_ROOT}

Config    : config/suites/stage2_pilot_12datasets_3seeds_all_methods.yaml
Data root : /kaggle/working/adaptive-fusion-ts2img/data/UCR
Base URL  : https://www.timeseriesclassification.com/aeon-toolkit
Overwrite : False
Datasets  : Coffee, ECG200, GunPoint, FordA, Wafer, ItalyPowerDemand, Trace, TwoLeadECG, CBF, Beef, Meat, OliveOil
------------------------------------------------------------------------
[downloaded] Coffee
  Train: /kaggle/working/adaptive-fusion-ts2img/data/UCR/Coffee/Coffee_TRAIN.tsv
  Test : /kaggle/working/adaptive-fusion-ts2img/data/UCR/Coffee/Coffee_TEST.tsv
[downloaded] ECG200
  Train: /kaggle/working/adaptive-fusion-ts2img/data/UCR/ECG200/ECG200_TRAIN.tsv
  Test : /kaggle/working/adaptive-fusion-ts2img/data/UCR/ECG200/ECG200_TEST.tsv
[downloaded] GunPoint
  Train: /kaggle/working/adaptive-fusion-ts2img/data/UCR/GunPoint/GunPoint_TRAIN.tsv
  Test : /kaggle/working/adaptive-fusion-ts2img/data/UCR/GunPoint/GunPoint_TEST.tsv
[downloaded] FordA
  Train: /kaggle/worki

## 6. Kiểm tra dữ liệu

In [9]:
STAGE2_DATASETS = [
    "Coffee",
    "ECG200",
    "GunPoint",
    "FordA",
    "Wafer",
    "ItalyPowerDemand",
    "Trace",
    "TwoLeadECG",
    "CBF",
    "Beef",
    "Meat",
    "OliveOil",
]

missing = []

for dataset in STAGE2_DATASETS:
    train_file = DATA_ROOT / dataset / f"{dataset}_TRAIN.tsv"
    test_file = DATA_ROOT / dataset / f"{dataset}_TEST.tsv"
    ok = train_file.is_file() and test_file.is_file()
    print(f"{dataset:20s}: {'OK' if ok else 'MISSING'}")
    if not ok:
        missing.append(dataset)

if missing:
    raise FileNotFoundError(f"Thiếu dataset: {missing}")

print("\nTất cả 12 datasets của Stage 2 đã sẵn sàng.")

Coffee              : OK
ECG200              : OK
GunPoint            : OK
FordA               : OK
Wafer               : OK
ItalyPowerDemand    : OK
Trace               : OK
TwoLeadECG          : OK
CBF                 : OK
Beef                : OK
Meat                : OK
OliveOil            : OK

Tất cả 12 datasets của Stage 2 đã sẵn sàng.


## 7. Tạo suite chạy theo chunk

In [10]:
import copy
import yaml

ORIGINAL_SUITE = PROJECT_DIR / "config" / "suites" / "stage2_pilot_12datasets_3seeds_all_methods.yaml"

CHUNKS = {
    1: ["Coffee", "ECG200", "FordA", "Beef"],
    2: ["GunPoint", "Wafer", "Trace", "Meat"],
    3: ["ItalyPowerDemand", "TwoLeadECG", "CBF", "OliveOil"],
}

with ORIGINAL_SUITE.open("r", encoding="utf-8") as f:
    stage2_suite = yaml.safe_load(f)

for chunk_id, datasets in CHUNKS.items():
    chunk_suite = copy.deepcopy(stage2_suite)
    chunk_suite["description"] = (
        f"Stage 2 Kaggle chunk {chunk_id}: "
        f"{len(datasets)} datasets, 3 seeds, 7 methods."
    )
    chunk_suite["datasets"] = datasets

    chunk_path = PROJECT_DIR / "config" / "suites" / f"stage2_kaggle_chunk{chunk_id}.yaml"
    with chunk_path.open("w", encoding="utf-8") as f:
        yaml.safe_dump(
            chunk_suite,
            f,
            sort_keys=False,
            allow_unicode=True,
        )

if RUN_MODE == "full":
    SUITE_FILE = ORIGINAL_SUITE
    SELECTED_DATASETS = STAGE2_DATASETS
    EXPECTED_SELECTED_RUNS = 252
elif RUN_MODE == "chunk":
    if CHUNK_ID not in CHUNKS:
        raise ValueError("CHUNK_ID phải là 1, 2 hoặc 3.")
    SUITE_FILE = PROJECT_DIR / "config" / "suites" / f"stage2_kaggle_chunk{CHUNK_ID}.yaml"
    SELECTED_DATASETS = CHUNKS[CHUNK_ID]
    EXPECTED_SELECTED_RUNS = 84
else:
    raise ValueError("RUN_MODE phải là 'chunk' hoặc 'full'.")

print("Suite             :", SUITE_FILE)
print("Selected datasets :", SELECTED_DATASETS)
print("Expected runs     :", EXPECTED_SELECTED_RUNS)

Suite             : /kaggle/working/adaptive-fusion-ts2img/config/suites/stage2_kaggle_chunk3.yaml
Selected datasets : ['ItalyPowerDemand', 'TwoLeadECG', 'CBF', 'OliveOil']
Expected runs     : 84


## 8. Dry run — chưa huấn luyện

In [11]:
!python -m src.cli.batch_run \
  --suite {SUITE_FILE} \
  --set \
      paths.data_root={DATA_ROOT} \
      paths.output_root={OUTPUT_ROOT} \
      paths.cache_root={CACHE_ROOT} \
      training.num_workers=2 \
  --dry-run

Running: /usr/bin/python3 -m src.train --config /tmp/adaptive_fusion_configs/ItalyPowerDemand_seed42_cnn1d.yaml
Running: /usr/bin/python3 -m src.train --config /tmp/adaptive_fusion_configs/ItalyPowerDemand_seed43_cnn1d.yaml
Running: /usr/bin/python3 -m src.train --config /tmp/adaptive_fusion_configs/ItalyPowerDemand_seed44_cnn1d.yaml
Running: /usr/bin/python3 -m src.train --config /tmp/adaptive_fusion_configs/TwoLeadECG_seed42_cnn1d.yaml
Running: /usr/bin/python3 -m src.train --config /tmp/adaptive_fusion_configs/TwoLeadECG_seed43_cnn1d.yaml
Running: /usr/bin/python3 -m src.train --config /tmp/adaptive_fusion_configs/TwoLeadECG_seed44_cnn1d.yaml
Running: /usr/bin/python3 -m src.train --config /tmp/adaptive_fusion_configs/CBF_seed42_cnn1d.yaml
Running: /usr/bin/python3 -m src.train --config /tmp/adaptive_fusion_configs/CBF_seed43_cnn1d.yaml
Running: /usr/bin/python3 -m src.train --config /tmp/adaptive_fusion_configs/CBF_seed44_cnn1d.yaml
Running: /usr/bin/python3 -m src.train --config /

In [12]:
from pathlib import Path
from itertools import product
import pandas as pd
import json
import re

# =========================================================
# 1. Các tên cột có thể được sử dụng trong file kết quả
# =========================================================
COLUMN_ALIASES = {
    "dataset": [
        "dataset", "dataset_name", "data_name"
    ],
    "method": [
        "experiment", "experiment_name", "method",
        "method_name", "model", "model_name", "approach"
    ],
    "seed": [
        "seed", "random_seed", "run_seed"
    ],
    "status": [
        "status", "run_status", "state"
    ]
}


def normalize_column_name(name):
    """Chuẩn hóa tên cột để dễ nhận diện."""
    name = str(name).strip().lower()
    name = re.sub(r"[^a-z0-9]+", "_", name)
    return name.strip("_")


def normalize_dataframe(df):
    df = df.copy()
    df.columns = [
        normalize_column_name(col)
        for col in df.columns
    ]
    return df


def find_column(columns, aliases):
    """Tìm cột theo danh sách tên thay thế."""
    columns = list(columns)

    # Ưu tiên tên trùng hoàn toàn
    for alias in aliases:
        if alias in columns:
            return alias

    # Sau đó tìm tên có chứa từ khóa
    for column in columns:
        for alias in aliases:
            if alias in column:
                return column

    return None


def convert_to_dataframes(source_name, obj):
    """Chuyển DataFrame/list/dict thành danh sách DataFrame."""
    outputs = []

    if isinstance(obj, pd.DataFrame):
        outputs.append((source_name, obj))

    elif isinstance(obj, list):
        if obj and isinstance(obj[0], dict):
            try:
                outputs.append((source_name, pd.DataFrame(obj)))
            except Exception:
                pass

    elif isinstance(obj, dict):
        # Dictionary dạng cột -> danh sách
        try:
            df = pd.DataFrame(obj)
            if not df.empty:
                outputs.append((source_name, df))
        except Exception:
            pass

        # Dictionary chứa records/runs/results
        for key, value in obj.items():
            if isinstance(value, list) and value:
                if isinstance(value[0], dict):
                    try:
                        outputs.append(
                            (f"{source_name}[{key}]", pd.DataFrame(value))
                        )
                    except Exception:
                        pass

    return outputs


# =========================================================
# 2. Tìm dữ liệu trong bộ nhớ notebook
# =========================================================
candidate_dataframes = []

for variable_name, obj in list(globals().items()):
    if variable_name.startswith("_"):
        continue

    try:
        candidate_dataframes.extend(
            convert_to_dataframes(
                f"Biến bộ nhớ: {variable_name}",
                obj
            )
        )
    except Exception:
        pass


# =========================================================
# 3. Tìm các file kết quả trong /kaggle/working
# =========================================================
search_roots = [Path("/kaggle/working")]

if "PROJECT_DIR" in globals():
    try:
        project_path = Path(PROJECT_DIR)
        if project_path.exists():
            search_roots.append(project_path)
    except Exception:
        pass

# Loại bỏ thư mục trùng
search_roots = list(dict.fromkeys(search_roots))

supported_extensions = {
    ".csv", ".json", ".jsonl", ".parquet"
}

result_keywords = {
    "result", "results", "metric", "metrics",
    "run", "runs", "progress", "history",
    "record", "records", "manifest", "summary"
}

searched_files = []

for root in search_roots:
    if not root.exists():
        continue

    for path in root.rglob("*"):
        if not path.is_file():
            continue

        if path.suffix.lower() not in supported_extensions:
            continue

        # Tránh đọc file quá lớn
        try:
            if path.stat().st_size > 100 * 1024 * 1024:
                continue
        except Exception:
            continue

        filename_lower = path.name.lower()

        # Ưu tiên các file có tên liên quan kết quả
        if not any(
            keyword in filename_lower
            for keyword in result_keywords
        ):
            continue

        searched_files.append(path)

        try:
            suffix = path.suffix.lower()

            if suffix == ".csv":
                df = pd.read_csv(path)

            elif suffix == ".parquet":
                df = pd.read_parquet(path)

            elif suffix == ".jsonl":
                df = pd.read_json(path, lines=True)

            elif suffix == ".json":
                with open(path, "r", encoding="utf-8") as file:
                    obj = json.load(file)

                converted = convert_to_dataframes(
                    f"File: {path}",
                    obj
                )

                candidate_dataframes.extend(converted)
                continue

            candidate_dataframes.append(
                (f"File: {path}", df)
            )

        except Exception:
            # Bỏ qua file không đọc được
            pass


# =========================================================
# 4. Chọn bảng có dataset, method và seed
# =========================================================
valid_candidates = []

for source_name, df in candidate_dataframes:
    if df is None or df.empty:
        continue

    normalized_df = normalize_dataframe(df)

    dataset_col = find_column(
        normalized_df.columns,
        COLUMN_ALIASES["dataset"]
    )

    method_col = find_column(
        normalized_df.columns,
        COLUMN_ALIASES["method"]
    )

    seed_col = find_column(
        normalized_df.columns,
        COLUMN_ALIASES["seed"]
    )

    status_col = find_column(
        normalized_df.columns,
        COLUMN_ALIASES["status"]
    )

    if dataset_col and method_col and seed_col:
        number_of_runs = (
            normalized_df[
                [dataset_col, method_col, seed_col]
            ]
            .drop_duplicates()
            .shape[0]
        )

        valid_candidates.append({
            "source": source_name,
            "df": normalized_df,
            "dataset_col": dataset_col,
            "method_col": method_col,
            "seed_col": seed_col,
            "status_col": status_col,
            "number_of_runs": number_of_runs
        })


if not valid_candidates:
    print("=" * 75)
    print("CHƯA TÌM THẤY FILE/BẢNG KẾT QUẢ CHI TIẾT")
    print("=" * 75)

    print("\nCác file có tên liên quan đến kết quả đã tìm thấy:")

    if searched_files:
        for path in searched_files[:100]:
            print(" -", path)
    else:
        print("Không tìm thấy CSV/JSON/Parquet phù hợp trong /kaggle/working.")

    print("\nCác biến đáng chú ý trong bộ nhớ:")

    keywords = [
        "result", "run", "metric", "record",
        "progress", "history", "summary"
    ]

    found_variable = False

    for variable_name, obj in list(globals().items()):
        if any(
            keyword in variable_name.lower()
            for keyword in keywords
        ):
            found_variable = True

            try:
                object_length = len(obj)
            except Exception:
                object_length = "không xác định"

            print(
                f" - {variable_name}: "
                f"{type(obj).__name__}, kích thước={object_length}"
            )

    if not found_variable:
        print("Không tìm thấy biến có tên liên quan.")

    raise RuntimeError(
        "Không tìm thấy dữ liệu chi tiết để kiểm tra từng chunk. "
        "Hãy xem danh sách file/biến được in phía trên."
    )


# Chọn nguồn có nhiều run duy nhất nhất
selected = max(
    valid_candidates,
    key=lambda item: item["number_of_runs"]
)

runs_df = selected["df"].copy()
dataset_col = selected["dataset_col"]
method_col = selected["method_col"]
seed_col = selected["seed_col"]
status_col = selected["status_col"]

print("=" * 75)
print("NGUỒN KẾT QUẢ ĐƯỢC SỬ DỤNG")
print("=" * 75)
print("Nguồn      :", selected["source"])
print("Số dòng    :", len(runs_df))
print("Dataset col:", dataset_col)
print("Method col :", method_col)
print("Seed col   :", seed_col)
print("Status col :", status_col)


# =========================================================
# 5. Loại bỏ các run đang lỗi/chưa hoàn thành
# =========================================================
if status_col is not None:
    status_text = (
        runs_df[status_col]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    failed_pattern = (
        r"fail|failed|error|exception|pending|running|"
        r"incomplete|cancelled|canceled"
    )

    failed_mask = status_text.str.contains(
        failed_pattern,
        regex=True,
        na=False
    )

    failed_runs_df = runs_df[failed_mask].copy()
    completed_df = runs_df[~failed_mask].copy()
else:
    failed_runs_df = pd.DataFrame()
    completed_df = runs_df.copy()


# Chuẩn hóa giá trị
completed_df[dataset_col] = (
    completed_df[dataset_col]
    .astype(str)
    .str.strip()
)

completed_df[method_col] = (
    completed_df[method_col]
    .astype(str)
    .str.strip()
)


def normalize_seed(value):
    """Tránh khác biệt giữa 42, 42.0 và '42'."""
    try:
        number = float(value)
        if number.is_integer():
            return str(int(number))
    except Exception:
        pass

    return str(value).strip()


completed_df[seed_col] = (
    completed_df[seed_col]
    .apply(normalize_seed)
)

completed_df = completed_df.drop_duplicates(
    subset=[dataset_col, method_col, seed_col]
)


# =========================================================
# 6. Xác định methods và seeds
# =========================================================
expected_methods = sorted(
    completed_df[method_col]
    .dropna()
    .unique()
    .tolist()
)

expected_seeds = sorted(
    completed_df[seed_col]
    .dropna()
    .unique()
    .tolist()
)

print("\nMethods:", expected_methods)
print("Seeds  :", expected_seeds)
print("Số run duy nhất hiện có:", len(completed_df))


# =========================================================
# 7. Kiểm tra từng chunk
# =========================================================
if "CHUNKS" not in globals():
    raise RuntimeError(
        "Không tìm thấy biến CHUNKS. "
        "Hãy chạy lại ô cấu hình chứa CHUNKS trước."
    )

actual_runs = set(
    completed_df[
        [dataset_col, method_col, seed_col]
    ].itertuples(index=False, name=None)
)

all_missing_rows = []

print("\n" + "=" * 75)
print("KẾT QUẢ KIỂM TRA TỪNG CHUNK")
print("=" * 75)

for chunk_id, chunk_datasets in CHUNKS.items():
    chunk_datasets = [
        str(dataset).strip()
        for dataset in chunk_datasets
    ]

    expected_runs = set(
        product(
            chunk_datasets,
            expected_methods,
            expected_seeds
        )
    )

    completed_runs = expected_runs.intersection(actual_runs)
    missing_runs = expected_runs.difference(actual_runs)

    print(
        f"\nChunk {chunk_id}: "
        f"{len(completed_runs)}/{len(expected_runs)} runs"
    )

    completely_missing_datasets = [
        dataset
        for dataset in chunk_datasets
        if dataset not in set(completed_df[dataset_col])
    ]

    if not missing_runs:
        print("  Trạng thái: ĐÃ ĐỦ")

    else:
        print("  Trạng thái: ĐANG THIẾU")

        if completely_missing_datasets:
            print(
                "  Dataset hoàn toàn chưa có:",
                completely_missing_datasets
            )

        missing_df = pd.DataFrame(
            sorted(missing_runs),
            columns=["dataset", "experiment", "seed"]
        )

        missing_summary = (
            missing_df
            .groupby("dataset")
            .size()
            .sort_values(ascending=False)
        )

        print("  Số run thiếu theo dataset:")

        for dataset, number_missing in missing_summary.items():
            print(
                f"    - {dataset}: thiếu {number_missing} runs"
            )

        for dataset, experiment, seed in missing_runs:
            all_missing_rows.append({
                "chunk": chunk_id,
                "dataset": dataset,
                "experiment": experiment,
                "seed": seed
            })


# =========================================================
# 8. Xuất danh sách run thiếu
# =========================================================
missing_output = pd.DataFrame(all_missing_rows)

print("\n" + "=" * 75)
print("TỔNG KẾT")
print("=" * 75)

if missing_output.empty:
    print("Tất cả các chunk đều đầy đủ.")

else:
    missing_output = missing_output.sort_values(
        ["chunk", "dataset", "experiment", "seed"]
    )

    output_path = Path(
        "/kaggle/working/missing_runs_by_chunk.csv"
    )

    missing_output.to_csv(
        output_path,
        index=False
    )

    print("Tổng số run thiếu:", len(missing_output))
    print("Danh sách run thiếu:", output_path)

    display(missing_output.head(100))


# =========================================================
# 9. Hiển thị các run ghi nhận trạng thái lỗi
# =========================================================
if not failed_runs_df.empty:
    print("\n" + "=" * 75)
    print("CÁC RUN CÓ TRẠNG THÁI LỖI/CHƯA HOÀN THÀNH")
    print("=" * 75)

    useful_columns = [
        column
        for column in [
            dataset_col,
            method_col,
            seed_col,
            status_col,
            "error",
            "exception",
            "message"
        ]
        if column is not None
        and column in failed_runs_df.columns
    ]

    display(failed_runs_df[useful_columns].head(100))

NGUỒN KẾT QUẢ ĐƯỢC SỬ DỤNG
Nguồn      : File: /kaggle/working/adaptive-fusion-ts2img/results/stage0_smoke_test/results_all_runs.csv
Số dòng    : 105
Dataset col: dataset
Method col : experiment
Seed col   : seed
Status col : None

Methods: ['adaptive_fusion_full', 'cnn1d', 'gaf_lightcnn', 'manual_feature_concat', 'mtf_lightcnn', 'rp_lightcnn', 'stft_lightcnn']
Seeds  : ['42', '43', '44']
Số run duy nhất hiện có: 105

KẾT QUẢ KIỂM TRA TỪNG CHUNK

Chunk 1: 63/84 runs
  Trạng thái: ĐANG THIẾU
  Dataset hoàn toàn chưa có: ['Beef']
  Số run thiếu theo dataset:
    - Beef: thiếu 21 runs

Chunk 2: 42/84 runs
  Trạng thái: ĐANG THIẾU
  Dataset hoàn toàn chưa có: ['Trace', 'Meat']
  Số run thiếu theo dataset:
    - Meat: thiếu 21 runs
    - Trace: thiếu 21 runs

Chunk 3: 0/84 runs
  Trạng thái: ĐANG THIẾU
  Dataset hoàn toàn chưa có: ['ItalyPowerDemand', 'TwoLeadECG', 'CBF', 'OliveOil']
  Số run thiếu theo dataset:
    - CBF: thiếu 21 runs
    - ItalyPowerDemand: thiếu 21 runs
    - OliveOil: t

,chunk,dataset,experiment,seed
14,1,Beef,adaptive_fusion_full,42
18,1,Beef,adaptive_fusion_full,43
0,1,Beef,adaptive_fusion_full,44
1,1,Beef,cnn1d,42
2,1,Beef,cnn1d,43
...,...,...,...,...
74,3,ItalyPowerDemand,manual_feature_concat,44
124,3,ItalyPowerDemand,mtf_lightcnn,42
145,3,ItalyPowerDemand,mtf_lightcnn,43
112,3,ItalyPowerDemand,mtf_lightcnn,44


In [13]:
# Các dataset đang xuất hiện trong bảng 168/252
PRESENT_DATASETS = {
    "Beef",
    "CBF",
    "Coffee",
    "ECG200",
    "FordA",
    "ItalyPowerDemand",
    "OliveOil",
    "TwoLeadECG",
}

print("=" * 70)
print("KIỂM TRA DATASET THEO TỪNG CHUNK")
print("=" * 70)

for chunk_id, datasets in CHUNKS.items():
    datasets = set(datasets)

    present = sorted(datasets & PRESENT_DATASETS)
    missing = sorted(datasets - PRESENT_DATASETS)

    completed_runs = len(present) * 7 * 3
    expected_runs = len(datasets) * 7 * 3

    print(f"\nChunk {chunk_id}: {completed_runs}/{expected_runs} runs")
    print("  Dataset đã có :", present)

    if missing:
        print("  Dataset còn thiếu:", missing)
    else:
        print("  Trạng thái: ĐÃ ĐỦ")

KIỂM TRA DATASET THEO TỪNG CHUNK

Chunk 1: 84/84 runs
  Dataset đã có : ['Beef', 'Coffee', 'ECG200', 'FordA']
  Trạng thái: ĐÃ ĐỦ

Chunk 2: 0/84 runs
  Dataset đã có : []
  Dataset còn thiếu: ['GunPoint', 'Meat', 'Trace', 'Wafer']

Chunk 3: 84/84 runs
  Dataset đã có : ['CBF', 'ItalyPowerDemand', 'OliveOil', 'TwoLeadECG']
  Trạng thái: ĐÃ ĐỦ


## 9. Chạy huấn luyện

Cell này có thể chạy lâu. Khi phiên bị ngắt, khôi phục `stage2_resume_bundle.tar.gz`, giữ nguyên `CHUNK_ID` và chạy lại. Pipeline sẽ dùng checkpoint để tiếp tục.

In [14]:
!python -m src.cli.batch_run \
  --suite {SUITE_FILE} \
  --set \
      paths.data_root={DATA_ROOT} \
      paths.output_root={OUTPUT_ROOT} \
      paths.cache_root={CACHE_ROOT} \
      training.num_workers=2

Running: /usr/bin/python3 -m src.train --config /tmp/adaptive_fusion_configs/ItalyPowerDemand_seed42_cnn1d.yaml
Dataset: ItalyPowerDemand
Experiment: cnn1d
Input mode: raw_1d
Run directory: /kaggle/working/adaptive_fusion_stage2/outputs/ItalyPowerDemand_cnn1d_seed42
Device: cuda
Epoch 001 | train_loss=0.7220 | train_f1=0.3291 | val_f1=0.3333 | val_acc=0.5000 | time=1.5s
Saved best model.
Epoch 002 | train_loss=0.6739 | train_f1=0.3291 | val_f1=0.3333 | val_acc=0.5000 | time=0.2s
Epoch 003 | train_loss=0.6643 | train_f1=0.6954 | val_f1=0.7083 | val_acc=0.7143 | time=0.2s
Saved best model.
Epoch 004 | train_loss=0.6338 | train_f1=0.5213 | val_f1=0.5906 | val_acc=0.6429 | time=0.2s
Epoch 005 | train_loss=0.6238 | train_f1=0.4176 | val_f1=0.4750 | val_acc=0.5714 | time=0.2s
Epoch 006 | train_loss=0.6173 | train_f1=0.3788 | val_f1=0.3333 | val_acc=0.5000 | time=0.2s
Epoch 007 | train_loss=0.5874 | train_f1=0.3788 | val_f1=0.3333 | val_acc=0.5000 | time=0.2s
Epoch 008 | train_loss=0.6020 | t

## 10. Kiểm tra tiến độ

In [15]:
import json
import pandas as pd
from IPython.display import display

records = []

for summary_path in sorted(OUTPUT_ROOT.rglob("summary.json")):
    with summary_path.open("r", encoding="utf-8") as f:
        records.append(json.load(f))

if not records:
    print("Chưa tìm thấy run hoàn thành.")
else:
    runs_df = pd.DataFrame(records)

    print("Runs hoàn thành :", len(runs_df), "/ 252")
    print("Datasets       :", runs_df["dataset"].nunique())
    print("Methods        :", runs_df["experiment"].nunique())
    print("Seeds          :", runs_df["seed"].nunique())

    coverage = (
        runs_df.groupby(["dataset", "experiment"])["seed"]
        .nunique()
        .unstack(fill_value=0)
    )
    display(coverage)

    selected_done = runs_df[runs_df["dataset"].isin(SELECTED_DATASETS)]
    selected_count = len(selected_done)
    print(
        f"Runs thuộc phần đang chọn: {selected_count} / {EXPECTED_SELECTED_RUNS}"
    )

Runs hoàn thành : 252 / 252
Datasets       : 12
Methods        : 7
Seeds          : 3


experiment,adaptive_fusion_full,cnn1d,gaf_lightcnn,manual_feature_concat,mtf_lightcnn,rp_lightcnn,stft_lightcnn
dataset,,,,,,,
Beef,3,3,3,3,3,3,3
CBF,3,3,3,3,3,3,3
Coffee,3,3,3,3,3,3,3
ECG200,3,3,3,3,3,3,3
FordA,3,3,3,3,3,3,3
GunPoint,3,3,3,3,3,3,3
ItalyPowerDemand,3,3,3,3,3,3,3
Meat,3,3,3,3,3,3,3
OliveOil,3,3,3,3,3,3,3


Runs thuộc phần đang chọn: 84 / 84


## 11. Tạo paper package sau khi đủ 252 runs

In [16]:
summary_files = list(OUTPUT_ROOT.rglob("summary.json"))
completed_runs = len(summary_files)

if completed_runs < 252:
    print(
        f"Hiện có {completed_runs}/252 runs. "
        "Hãy hoàn thành các chunk còn lại trước khi tạo paper package cuối."
    )
else:
    PAPER_DIR = OUTPUT_ROOT / "paper_package"

    !python -m src.cli.make_paper_package \
      --output-root {OUTPUT_ROOT} \
      --results-csv {OUTPUT_ROOT}/summary_all_runs.csv \
      --out-dir {PAPER_DIR} \
      --metric test_macro_f1 \
      --proposed adaptive_fusion_full

    print("Paper package:", PAPER_DIR)

Paper package saved to: /kaggle/working/adaptive_fusion_stage2/outputs/paper_package
Runs: 252; datasets: 12; methods: 7
Main files:
 - /kaggle/working/adaptive_fusion_stage2/outputs/paper_package/summary_all_runs.csv
 - /kaggle/working/adaptive_fusion_stage2/outputs/paper_package/average_ranks_test_macro_f1.csv
 - /kaggle/working/adaptive_fusion_stage2/outputs/paper_package/critical_difference_test_macro_f1.png
 - /kaggle/working/adaptive_fusion_stage2/outputs/paper_package/table_ablation_summary_test_macro_f1.csv
 - /kaggle/working/adaptive_fusion_stage2/outputs/paper_package/README.md
Paper package: /kaggle/working/adaptive_fusion_stage2/outputs/paper_package


## 12. Nén bảng kết quả và paper package

In [17]:
import tarfile

PAPER_ARCHIVE = Path("/kaggle/working/stage2_paper_package.tar.gz")
PAPER_DIR = OUTPUT_ROOT / "paper_package"
SUMMARY_CSV = OUTPUT_ROOT / "summary_all_runs.csv"

if PAPER_DIR.exists() and SUMMARY_CSV.exists():
    with tarfile.open(PAPER_ARCHIVE, "w:gz") as tar:
        tar.add(PAPER_DIR, arcname="paper_package")
        tar.add(SUMMARY_CSV, arcname="summary_all_runs.csv")
    print("Created:", PAPER_ARCHIVE)
else:
    print("Paper package chưa tồn tại. Chỉ tạo archive này sau khi đủ 252 runs.")

Created: /kaggle/working/stage2_paper_package.tar.gz


## 13. Tạo gói resume để chuyển sang phiên Kaggle tiếp theo — tùy chọn

Gói này chứa outputs, checkpoints và cache. Sau khi cell hoàn thành:

1. chọn **Save Version** để lưu file đầu ra;
2. ở phiên sau, Add Data chứa `stage2_resume_bundle.tar.gz`;
3. chạy lại notebook với `AUTO_RESTORE=True`;
4. đổi `CHUNK_ID` sang phần tiếp theo.

In [18]:
RESUME_ARCHIVE = Path("/kaggle/working/stage2_resume_bundle.tar.gz")

with tarfile.open(RESUME_ARCHIVE, "w:gz") as tar:
    tar.add(RUN_ROOT, arcname=RUN_ROOT.name)

print("Created:", RESUME_ARCHIVE)
print("Kích thước có thể lớn vì chứa checkpoints và cache ảnh 2D.")

Created: /kaggle/working/stage2_resume_bundle.tar.gz
Kích thước có thể lớn vì chứa checkpoints và cache ảnh 2D.


## Thứ tự chạy khuyến nghị

```text
Phiên 1: RUN_MODE="chunk", CHUNK_ID=1
-> train
-> kiểm tra tiến độ
-> tạo stage2_resume_bundle.tar.gz
-> Save Version

Phiên 2: Add Data từ output phiên 1
-> AUTO_RESTORE=True
-> CHUNK_ID=2
-> train
-> tạo lại resume bundle
-> Save Version

Phiên 3: Add Data từ output phiên 2
-> AUTO_RESTORE=True
-> CHUNK_ID=3
-> train
-> kiểm tra đủ 252 runs
-> tạo paper package
```

Không thêm `--no-resume` khi muốn tiếp tục từ checkpoint.